# Ejercicio Práctico: Web Scraping de un E-Commerce de Libros (ETL con BeautifulSoup y Pandas)

En este ejercicio vas a construir por tu cuenta un flujo completo de **Extracción, Transformación y Carga (ETL)** utilizando `requests`, `BeautifulSoup` y `pandas` sobre la tienda online de práctica [Books to Scrape](https://books.toscrape.com/).

### Objetivo del ejercicio
Extraer información de múltiples categorías de libros para construir un `DataFrame` limpio con **4 variables clave por libro**:
1. **Título completo** (`str`)
2. **Precio en libras** (`float`, limpiando el símbolo `£`)
3. **Valoración en estrellas** (`str`, extraída de los atributos de clase)
4. **Disponibilidad en Stock** (`str`, limpiando saltos de línea y espacios)

---

## Paso 1: Importación de librerías y configuración

Importa las librerías necesarias (`time`, `requests`, `BeautifulSoup`, `os` y `pandas`) y define el diccionario `headers` con el `User-Agent`.

Setea el máximo de caracteres a 800 en pandas

In [1]:
import time
import requests
from bs4 import BeautifulSoup as bs
import os 
import pandas as pd

# Configuramos Pandas para que muestre textos largos sin cortarlos
pd.set_option("max_colwidth", 800)

# Definimos nuestro 'disfraz' de navegador (User-Agent) para evitar bloqueos 403
headers= {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

## Paso 2: Regla del Francotirador (Inspeccionar y probar con 1 solo libro)

Antes de crear ningún bucle, descarga la página principal `https://books.toscrape.com/`, abre el Inspector del navegador (`F12`) y localiza la estructura HTML de un libro:

In [ ]:
# 1. Hacemos la petición de prueba a la portada de la tiendaa
url="https://books.toscrape.com/"

# Verificamos que responde 200 OK
response=requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

# 2. Creamos la sopa
soup=bs(response.content, "html.parser")

# 3. Buscamos todas las tarjetas de libros y seleccionamos el primero (índice 0) para probar
cards=soup.find_all("article", class_="product_pod")
print(f"la cantidad de libros es: {len(cards)}")

primer_libro=cards[0]


la cantidad de libros es: 20


In [ ]:
# Probamos la extracción y limpieza de los 4 datos sobre 'primer_libro'
# 1. TÍTULO
titulo=primer_libro.find("h3").find("a")["title"]
# titulo=primer_libro.find("h3").find("a").get_attribute_list("title")
#titulo=primer_libro.find("h3").find("a").get_text()

# 2. PRECIO: Quitamos el símbolo '£' y convertimos a float
precio=float(primer_libro.find("p", class_="price_color").get_text().replace("£", "").strip())

# 3. ESTRELLAS
estrellas=primer_libro.find("p", class_="star-rating")["class"][1]

# 4. STOCK: Usamos strip=True en get_text() para limpiar los saltos de línea
stock=primer_libro.find("p", class_="instock availability").get_text(strip=True)

print(f"Título:    {titulo}")
print(f"Precio:    {precio} ({type(precio).__name__})")
print(f"Estrellas: {estrellas}")
print(f"Stock:     {stock}")

Título:    A Light in the Attic
Precio:    51.77 (float)
Estrellas: Three
Stock:     In stock


## Paso 3: Extracción Multi-Página con Bucle y `time.sleep`

Ahora que sabemos extraer los 4 datos de una tarjeta, vamos a recorrer **3 categorías distintas de la tienda** (*Mystery*, *Science Fiction* y *Travel*) y a extraer los primeros 10 libros de cada una.

* **Regla ética y técnica:** Recuerda añadir `time.sleep(1)` al final de cada página para no saturar el servidor.

In [34]:
# Lista de tuplas con (Nombre_Categoria, URL) a scrapear
urls_categorias = [
    ("Misterio",        "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"),
    ("Ciencia Ficción", "https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html"),
    ("Viajes",          "https://books.toscrape.com/catalogue/category/books/travel_2/index.html")
]

# Inicializamos el diccionario vacío donde acumularemos los datos de todas las páginas
datos_libros = {
    "Categoria": [],
    "Titulo": [],
    "Precio_Libras": [],
    "Estrellas": [],
    "Stock": []
}

# BUCLE EXTERIOR: recorre cada categoría
for categoria,url in urls_categorias:
    print(f"Descargando página de categoria: {categoria} ({url})...")
    # Bloque fijo de petición y BeautifulSoup
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup_pagina = bs(response.content, "html.parser")

    libros=soup_pagina.find_all("article", class_="product_pod")[:10]

    # BUCLE INTERIOR: recorre los primeros 10 libros de esa categoría
    for libro in libros: 
        # 1. Título
        titulo=libro.find("h3").find("a")["title"]

        # 2. Precio
        precio=float(libro.find("p", class_="price_color").get_text().replace("£", "").strip())    

        # 3. Estrellas
        estrellas=libro.find("p", class_="star-rating")["class"][1]    

        # 4. Stock
        stock=libro.find("p", class_="instock availability").get_text(strip=True)    

        # Guardamos los datos en las listas de nuestro diccionario
        datos_libros["Categoria"].append(categoria)
        datos_libros["Titulo"].append(titulo)
        datos_libros["Precio_Libras"].append(precio)
        datos_libros["Estrellas"].append(estrellas)
        datos_libros["Stock"].append(stock)


    # Esperamos 1 segundo antes de pedir la siguiente página
    time.sleep(2)

print("¡Extracción completada con éxito!")
print(datos_libros)

Descargando página de categoria: Misterio (https://books.toscrape.com/catalogue/category/books/mystery_3/index.html)...
Descargando página de categoria: Ciencia Ficción (https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html)...
Descargando página de categoria: Viajes (https://books.toscrape.com/catalogue/category/books/travel_2/index.html)...
¡Extracción completada con éxito!
{'Categoria': ['Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes'], 'Titulo': ['Sharp Objects', 'In a Dark, Dark Wood', 'The Past Never Ends', 'A Murder in Time', 'The Murder of Roger Ackroyd (Hercule Poirot #4)', 'The Last Mile (Amos 

## Paso 4: Creación del DataFrame, Exploración y Guardado en CSV

1. Convierte el diccionario `datos_libros` en un `DataFrame` de Pandas llamado `df_libros`.
2. Comprueba sus dimensiones (`.shape`) y tipos de datos (`.dtypes`).
3. Calcula el **precio medio por categoría** usando `.groupby()`.
4. Guarda el resultado en `data/libros_scrapeados.csv` con separador `;` y sin índice (`index=False`).

In [39]:
# 1. Creamos el DataFrame a partir del diccionario recolectado
df=pd.DataFrame(datos_libros)

# 2. Comprobamos dimensiones y tipos de datos
print(df.shape)

# Mostramos los 10 primeros elementos del DataFrame
print(df.head(10))

(30, 5)
  Categoria                                           Titulo  Precio_Libras  \
0  Misterio                                    Sharp Objects          47.82   
1  Misterio                             In a Dark, Dark Wood          19.63   
2  Misterio                              The Past Never Ends          56.50   
3  Misterio                                 A Murder in Time          16.64   
4  Misterio  The Murder of Roger Ackroyd (Hercule Poirot #4)          44.10   
5  Misterio                   The Last Mile (Amos Decker #2)          54.21   
6  Misterio           That Darkness (Gardiner and Renner #1)          13.92   
7  Misterio             Tastes Like Fear (DI Marnie Rome #3)          10.69   
8  Misterio           A Time of Torment (Charlie Parker #14)          48.35   
9  Misterio          A Study in Scarlet (Sherlock Holmes #1)          16.73   

  Estrellas     Stock  
0      Four  In stock  
1       One  In stock  
2      Four  In stock  
3       One  In stock  
4 

In [42]:
import numpy as np

In [48]:
# 3. Precio medio, mínimo y máximo por categoría usando .groupby()
df.groupby("Categoria")["Precio_Libras"].agg(["mean", min, max])

,mean,min,max
Categoria,,,
Ciencia Ficción,28.584,10.65,48.74
Misterio,32.859,10.69,56.50
Viajes,41.166,23.21,56.88


In [49]:
# 4. Guardamos el DataFrame en un archivo CSV limpio usando la librería 'os'
df.to_csv("data/libros.csv")